In [3]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as mcm
from matplotlib.colors import Normalize
from concurrent.futures import ThreadPoolExecutor  # 改用多线程，完美兼容 Jupyter

# ==========================================
# 1. 配置参数与路径
# ==========================================
PROJ_STR = "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105"

SHP_PATH = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/city_shp/shi_en.shp")
CHINA_SHP = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国底图-中图社审过版本/中国底图/中国面.shp")
CHINA_SHP2 = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国国界线/九段线/九段线和群岛.shp")
POP_ROOT = Path('/Volumes/UCL/论文工作/空气污染/pop_city_revised')
OUTFILE = Path('/Users/shirley/Desktop/plots_V2/age_population_rates.csv')
                  
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 10,
    "axes.titleweight": "bold",
})

# 南海子图在 EPSG:4326 下的范围
_NANHAI_BOUNDS = (105, 2, 122, 24)

# ==========================================
# 2. 高效提取函数（路径 B：仅提取绝对人数）
# ==========================================
def _process_single_excel_path_b(args):
    """单文件处理函数：只提取总人数和老人绝对人数，不做任何比率计算"""
    city, dev, file_path, years = args
    try:
        male_df = pd.read_excel(file_path, sheet_name='male')
        female_df = pd.read_excel(file_path, sheet_name='female')
    except Exception:
        return []  # 读取失败则跳过

    file_records = []
    for year in years:
        if year not in male_df.columns or year not in female_df.columns:
            continue

        # 计算总人数
        total_pop = male_df[year].sum() + female_df[year].sum()
        
        # 计算 65+ 绝对人数
        pop_65_male = male_df.loc[male_df['Age'] >= 65, year].sum()
        pop_65_female = female_df.loc[female_df['Age'] >= 65, year].sum()
        pop_65plus = pop_65_male + pop_65_female

        file_records.append({
            'city': city,
            'dev_scenario': dev,
            'year': year,
            'pop_total_raw': total_pop,     # 记录总人口绝对值
            'pop_65plus_raw': pop_65plus    # 记录老人绝对值
        })
    return file_records

# ==========================================
# 3. 地理数据读取与数据清洗
# ==========================================
city_shp_raw = gpd.read_file(SHP_PATH)
china_border = gpd.read_file(CHINA_SHP).to_crs(PROJ_STR)
jiudanline = gpd.read_file(CHINA_SHP2).to_crs(PROJ_STR)

# 自动识别城市英文字段
for col in ['English', 'City', 'city']:
    if col in city_shp_raw.columns:
        city_key = col
        break
else:
    raise KeyError(f"Shapefile 不包含城市英文名列。可用列: {list(city_shp_raw.columns)}")

# 排除空值并清洗字符串
city_shp_raw = city_shp_raw[city_shp_raw[city_key].notna()].copy()
city_shp_raw[city_key] = city_shp_raw[city_key].astype(str).str.strip()
city_shp = city_shp_raw.to_crs(PROJ_STR)
valid_cities = city_shp[city_key].unique()

# ==========================================
# 4. 并行加速读取（路径 B 逻辑）
# ==========================================
dev_scenarios = [
    'SSPFer1_SSPMigr1', 'SSPFer1_SSPMigr2', 'SSPFer1_SSPMigr3',
    'SSPFer2_SSPMigr1', 'SSPFer2_SSPMigr2', 'SSPFer2_SSPMigr3',
    'SSPFer3_SSPMigr1', 'SSPFer3_SSPMigr2', 'SSPFer3_SSPMigr3',
    'SSPFer4_SSPMigr1', 'SSPFer4_SSPMigr2', 'SSPFer4_SSPMigr3',
    'SSPFer5_SSPMigr1', 'SSPFer5_SSPMigr2', 'SSPFer5_SSPMigr3'
]
years = [2020, 2060]

# 一次性扫描目录，提升 I/O 效率
existing_files = {f.name for f in POP_ROOT.glob("*.xlsx")}

# 构建多线程任务队列
tasks = []
for dev in dev_scenarios:
    for city in valid_cities:
        filename = f"{city}_{dev}.xlsx"
        if filename in existing_files:
            tasks.append((city, dev, POP_ROOT / filename, years))

records = []
if tasks:
    print(f"开始并行解析 {len(tasks)} 个 Excel 文件...")
    # 在 Jupyter 中使用 ThreadPoolExecutor 安全且高速
    with ThreadPoolExecutor() as executor:
        results = executor.map(_process_single_excel_path_b, tasks)
        for res in results:
            records.extend(res)
else:
    print("未在目录中找到匹配的 Excel 文件。")

pop_age_df = pd.DataFrame(records)

# ==========================================
# 5. 路径 B 的矩阵化聚合计算（核心变动）
# ==========================================
# 先对各个情景的人数绝对值求和/平均
avg_df = (
    pop_age_df
    .groupby(['city', 'year'], as_index=False)[['pop_total_raw', 'pop_65plus_raw']]
    .sum()  # 将 15 个情景下的人数各自相加
)

# 统一在外部进行一次除法，计算整体老龄化率
avg_df['mean_prop_65plus'] = np.where(
    avg_df['pop_total_raw'] > 0, 
    avg_df['pop_65plus_raw'] / avg_df['pop_total_raw'], 
    np.nan
)
avg_df.to_csv(OUTFILE, index=False)


开始并行解析 5400 个 Excel 文件...
